In [ ]:
import pandas as pd

def calculate_error_with_zero_handling(actual, predicted):
    if abs(actual) < 1e-6 and abs(predicted) < 1e-6:
        return 0.0 
    elif actual != 0:
        return abs(actual - predicted) / actual * 100
    return float('inf') 

df1 = pd.read_csv('G034_G035_combined.csv')
df2 = pd.read_csv('G036_predictions.csv')

final_results = []
for _, row1 in df1.iterrows():
    for _, row2 in df2.iterrows():
        if row1['cohort'] == row2['cohort'] and row1['elapsed_months'] == row2['elapsed_months']:
            if pd.isnull(row1['total_cac']) or pd.isnull(row2['total_cac']) or pd.isnull(row1['absolute_revenue_actual']) or pd.isnull(row2['absolute_revenue_actual']):
                continue
            cac_error = calculate_error_with_zero_handling(row1['total_cac'], row2['total_cac'])
            revenue_error = calculate_error_with_zero_handling(row1['absolute_revenue_actual'], row2['absolute_revenue_actual'])
            match = "Yes" if cac_error <= 2 and revenue_error <= 2 else "No"
            final_results.append({
                'cohort': row1['cohort'],
                'elapsed_months': row1['elapsed_months'],
                'percent_error_ltv_cac': cac_error,
                'percent_error_absolute_revenue_actual': revenue_error,
                'match': match
            })

final_results_df = pd.DataFrame(final_results)
final_results_df.to_csv('36_data_check.csv', index=False)
print("Final cohort error analysis saved'.")


In [ ]:
df1 = pd.read_csv('G027_G028_G029_combined.csv')
df2 = pd.read_csv('G030_predictions.csv')

final_results = []
for _, row1 in df1.iterrows():
    for _, row2 in df2.iterrows():
        if row1['cohort'] == row2['cohort'] and row1['elapsed_months'] == row2['elapsed_months']:
            if pd.isnull(row1['total_cac']) or pd.isnull(row2['total_cac']) or pd.isnull(row1['absolute_revenue_actual']) or pd.isnull(row2['absolute_revenue_actual']):
                continue
            cac_error = calculate_error_with_zero_handling(row1['total_cac'], row2['total_cac'])
            revenue_error = calculate_error_with_zero_handling(row1['absolute_revenue_actual'], row2['absolute_revenue_actual'])
            match = "Yes" if cac_error <= 2 and revenue_error <= 2 else "No"
            final_results.append({
                'cohort': row1['cohort'],
                'elapsed_months': row1['elapsed_months'],
                'percent_error_ltv_cac': cac_error,
                'percent_error_absolute_revenue_actual': revenue_error,
                'match': match
            })

final_results_df = pd.DataFrame(final_results)
final_results_df.to_csv('30_data_check.csv', index=False)
print("Final cohort error analysis saved.")


In [ ]:

df_g027 = pd.read_csv('G027_Dataset_12_2024.csv')
df_g027.columns

In [ ]:
import pandas as pd

# ========== STEP 1: READ G027, G028, G029 WITH SUFFIXES ==========

df_g027 = pd.read_csv('G027_Dataset_12_2024.csv')
df_g028 = pd.read_csv('G028_Dataset_12_2024.csv')
df_g029 = pd.read_csv('G029_Dataset_12_2024.csv')

df_g027 = df_g027.add_suffix('_g027')
df_g028 = df_g028.add_suffix('_g028')
df_g029 = df_g029.add_suffix('_g029')

# Rename the merge keys back to standard
df_g027.rename(columns={
    'cohort_g027': 'cohort',
    'elapsed_months_g027': 'elapsed_months'
}, inplace=True)

df_g028.rename(columns={
    'cohort_g028': 'cohort',
    'elapsed_months_g028': 'elapsed_months'
}, inplace=True)

df_g029.rename(columns={
    'cohort_g029': 'cohort',
    'elapsed_months_g029': 'elapsed_months'
}, inplace=True)

# ========== STEP 2: MERGE G027, G028, G029 ==========

merged_27_28 = pd.merge(
    df_g027, df_g028,
    on=['cohort', 'elapsed_months'],
    how='outer'
)
merged_27_28_29 = pd.merge(
    merged_27_28, df_g029,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in merged_27_28_29.columns:
    if merged_27_28_29[col].dtype in [float, int]:
        merged_27_28_29[col].fillna(0, inplace=True)
    else:
        merged_27_28_29[col].fillna('', inplace=True)

# ========== STEP 3: RENAME FOR ONLY THE COLUMNS WE NEED ==========

numeric_columns = ['net_revenue', 'total_cac']

for col in numeric_columns:
    old_27 = col + '_g027'
    old_28 = col + '_g028'
    old_29 = col + '_g029'
    new_27 = col + '_27'
    new_28 = col + '_28'
    new_29 = col + '_29'
    if old_27 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_27: new_27}, inplace=True)
    if old_28 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_28: new_28}, inplace=True)
    if old_29 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_29: new_29}, inplace=True)

# ========== STEP 4: READ & MERGE G030 ==========

df_g030 = pd.read_csv('G030_Dataset_12_2024.csv')
df_g030 = df_g030.add_suffix('_g030')

df_g030.rename(columns={
    'cohort_g030': 'cohort',
    'elapsed_months_g030': 'elapsed_months'
}, inplace=True)

final_merged = pd.merge(
    merged_27_28_29,
    df_g030,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in final_merged.columns:
    if final_merged[col].dtype in [float, int]:
        final_merged[col].fillna(0, inplace=True)
    else:
        final_merged[col].fillna('', inplace=True)

# Rename X_g030 -> X_30 for the numeric columns
for col in numeric_columns:
    old_30 = col + '_g030'
    new_30 = col + '_30'
    if old_30 in final_merged.columns:
        final_merged.rename(columns={old_30: new_30}, inplace=True)

# ========== STEP 5: CREATE SUM COLUMNS FOR (27 + 28 + 29) ==========

for col in numeric_columns:
    col_27 = col + '_27'
    col_28 = col + '_28'
    col_29 = col + '_29'
    sum_col = col + '_27_28_29_sum'
    final_merged[sum_col] = (
        final_merged[col_27] +
        final_merged[col_28] +
        final_merged[col_29]
    )

# ========== STEP 6: BUILD THE FINAL LIST OF COLUMNS ==========

# We'll keep: cohort, elapsed_months, current_month
# plus absolute_revenue_actual_27, absolute_revenue_actual_28, absolute_revenue_actual_29,
# absolute_revenue_actual_27_28_29_sum, absolute_revenue_actual_30,
# total_cac_27, total_cac_28, total_cac_29, total_cac_27_28_29_sum, total_cac_30

output_columns = ['cohort', 'elapsed_months']

for col in numeric_columns:
    # e.g. for 'absolute_revenue_actual', we add:
    # absolute_revenue_actual_27, absolute_revenue_actual_28, absolute_revenue_actual_29,
    # absolute_revenue_actual_27_28_29_sum, absolute_revenue_actual_30
    output_columns.append(col + '_27')
    output_columns.append(col + '_28')
    output_columns.append(col + '_29')
    output_columns.append(col + '_27_28_29_sum')
    output_columns.append(col + '_30')

final_output = final_merged[output_columns].copy()

# ========== STEP 7: SAVE THE FINAL CSV ==========

final_output.to_csv('G027_G028_G029_G030_values.csv', index=False)
print("Done! Check 'G027_G028_G029_G030_values.csv' for the combined values.")


In [ ]:
import pandas as pd

# ========== STEP 1: READ G034, G035 WITH SUFFIXES ==========

df_g034 = pd.read_csv('G034_Dataset_12_2024.csv')
df_g035 = pd.read_csv('G035_Dataset_12_2024.csv')

df_g034 = df_g034.add_suffix('_g034')
df_g035 = df_g035.add_suffix('_g035')

# Rename the merge keys back to standard
df_g034.rename(columns={
    'cohort_g034': 'cohort',
    'elapsed_months_g034': 'elapsed_months'
}, inplace=True)

df_g035.rename(columns={
    'cohort_g035': 'cohort',
    'elapsed_months_g035': 'elapsed_months'
}, inplace=True)

# ========== STEP 2: MERGE G034, G035 ==========

merged_34_35 = pd.merge(
    df_g034, df_g035,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in merged_34_35.columns:
    if merged_34_35[col].dtype in [float, int]:
        merged_34_35[col].fillna(0, inplace=True)
    else:
        merged_34_35[col].fillna('', inplace=True)

# ========== STEP 3: RENAME FOR ONLY THE COLUMNS WE NEED ==========

numeric_columns = ['net_revenue', 'total_cac']

for col in numeric_columns:
    old_34 = col + '_g034'
    old_35 = col + '_g035'
    new_34 = col + '_34'
    new_35 = col + '_35'
    if old_34 in merged_34_35.columns:
        merged_34_35.rename(columns={old_34: new_34}, inplace=True)
    if old_35 in merged_34_35.columns:
        merged_34_35.rename(columns={old_35: new_35}, inplace=True)

# ========== STEP 4: READ & MERGE G036 ==========

df_g036 = pd.read_csv('G036_Dataset_12_2024.csv')
df_g036 = df_g036.add_suffix('_g036')

df_g036.rename(columns={
    'cohort_g036': 'cohort',
    'elapsed_months_g036': 'elapsed_months'
}, inplace=True)

final_merged = pd.merge(
    merged_34_35,
    df_g036,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in final_merged.columns:
    if final_merged[col].dtype in [float, int]:
        final_merged[col].fillna(0, inplace=True)
    else:
        final_merged[col].fillna('', inplace=True)

# Rename X_g036 -> X_36 for the numeric columns
for col in numeric_columns:
    old_36 = col + '_g036'
    new_36 = col + '_36'
    if old_36 in final_merged.columns:
        final_merged.rename(columns={old_36: new_36}, inplace=True)

# ========== STEP 5: CREATE SUM COLUMNS FOR (34 + 35) ==========

for col in numeric_columns:
    col_34 = col + '_34'
    col_35 = col + '_35'
    sum_col = col + '_34_35_sum'
    final_merged[sum_col] = (
        final_merged[col_34] +
        final_merged[col_35]
    )

# ========== STEP 6: BUILD THE FINAL LIST OF COLUMNS ==========

output_columns = ['cohort', 'elapsed_months']

# For each numeric column, e.g. net_revenue_34, net_revenue_35, net_revenue_34_35_sum, net_revenue_36
for col in numeric_columns:
    output_columns.append(col + '_34')
    output_columns.append(col + '_35')
    output_columns.append(col + '_34_35_sum')
    output_columns.append(col + '_36')

final_output = final_merged[output_columns].copy()

# ========== STEP 7: SAVE THE FINAL CSV ==========

final_output.to_csv('G034_G035_G036_values.csv', index=False)
print("Done! Check 'G034_G035_G036_values.csv' for the combined values.")


In [ ]:
import pandas as pd

# ========== STEP 1: READ G027, G028, G029 WITH SUFFIXES ==========

df_g027 = pd.read_csv('G027_Dataset_12_2024.csv')
df_g028 = pd.read_csv('G028_Dataset_12_2024.csv')
df_g029 = pd.read_csv('G029_Dataset_12_2024.csv')

df_g027 = df_g027.add_suffix('_g027')
df_g028 = df_g028.add_suffix('_g028')
df_g029 = df_g029.add_suffix('_g029')

# Rename the merge keys back to standard
df_g027.rename(columns={
    'cohort_g027': 'cohort',
    'elapsed_months_g027': 'elapsed_months'
}, inplace=True)

df_g028.rename(columns={
    'cohort_g028': 'cohort',
    'elapsed_months_g028': 'elapsed_months'
}, inplace=True)

df_g029.rename(columns={
    'cohort_g029': 'cohort',
    'elapsed_months_g029': 'elapsed_months'
}, inplace=True)

# ========== STEP 2: MERGE G027, G028, G029 ==========

merged_27_28 = pd.merge(
    df_g027, df_g028,
    on=['cohort', 'elapsed_months'],
    how='outer'
)
merged_27_28_29 = pd.merge(
    merged_27_28, df_g029,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in merged_27_28_29.columns:
    if merged_27_28_29[col].dtype in [float, int]:
        merged_27_28_29[col].fillna(0, inplace=True)
    else:
        merged_27_28_29[col].fillna('', inplace=True)

# ========== STEP 3: RENAME FOR ONLY THE COLUMNS WE NEED ==========

numeric_columns = ['net_revenue', 'total_cac', 'total_users', 'transaction_period_ltv_cac']

for col in numeric_columns:
    old_27 = col + '_g027'
    old_28 = col + '_g028'
    old_29 = col + '_g029'
    new_27 = col + '_27'
    new_28 = col + '_28'
    new_29 = col + '_29'
    if old_27 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_27: new_27}, inplace=True)
    if old_28 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_28: new_28}, inplace=True)
    if old_29 in merged_27_28_29.columns:
        merged_27_28_29.rename(columns={old_29: new_29}, inplace=True)

# ========== STEP 4: READ & MERGE G030 ==========

df_g030 = pd.read_csv('G030_Dataset_12_2024.csv')
df_g030 = df_g030.add_suffix('_g030')

df_g030.rename(columns={
    'cohort_g030': 'cohort',
    'elapsed_months_g030': 'elapsed_months'
}, inplace=True)

final_merged = pd.merge(
    merged_27_28_29,
    df_g030,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in final_merged.columns:
    if final_merged[col].dtype in [float, int]:
        final_merged[col].fillna(0, inplace=True)
    else:
        final_merged[col].fillna('', inplace=True)

# Rename X_g030 -> X_30 for the numeric columns
for col in numeric_columns:
    old_30 = col + '_g030'
    new_30 = col + '_30'
    if old_30 in final_merged.columns:
        final_merged.rename(columns={old_30: new_30}, inplace=True)

# ========== STEP 5: CREATE SUM COLUMNS FOR (27 + 28 + 29) ==========

for col in ['net_revenue', 'total_cac', 'total_users']:
    col_27 = col + '_27'
    col_28 = col + '_28'
    col_29 = col + '_29'
    sum_col = col + '_27_28_29_sum'
    final_merged[sum_col] = (
        final_merged[col_27] +
        final_merged[col_28] +
        final_merged[col_29]
    )

# Calculate the combined ratio for 'transaction_period_ltv_cac' (net_revenue / total_cac)
final_merged['transaction_period_ltv_cac_27_28_29'] = (
    final_merged['net_revenue_27_28_29_sum'] / 
    final_merged['total_cac_27_28_29_sum']
).replace([float('inf'), -float('inf')], 0).fillna(0)

# ========== STEP 6: BUILD THE FINAL LIST OF COLUMNS ==========

output_columns = ['cohort', 'elapsed_months']

for col in numeric_columns:
    # e.g. for 'net_revenue', we add:
    # net_revenue_27, net_revenue_28, net_revenue_29,
    # net_revenue_27_28_29_sum, net_revenue_30
    if col != 'transaction_period_ltv_cac':  # Sum-based columns
        output_columns.append(col + '_27')
        output_columns.append(col + '_28')
        output_columns.append(col + '_29')
        output_columns.append(col + '_27_28_29_sum')
        output_columns.append(col + '_30')
    else:  # For ratio-based columns
        output_columns.append(col + '_27_28_29')
        output_columns.append(col + '_30')

final_output = final_merged[output_columns].copy()

# ========== STEP 7: SAVE THE FINAL CSV ==========

final_output.to_csv('G027_G028_G029_G030_values.csv', index=False)
print("Done! Check 'G027_G028_G029_G030_values.csv' for the combined values.")


In [ ]:
import pandas as pd

# ========== STEP 1: READ G034, G035 WITH SUFFIXES ==========

df_g034 = pd.read_csv('G034_Dataset_12_2024.csv')
df_g035 = pd.read_csv('G035_Dataset_12_2024.csv')

df_g034 = df_g034.add_suffix('_g034')
df_g035 = df_g035.add_suffix('_g035')

# Rename the merge keys back to standard
df_g034.rename(columns={
    'cohort_g034': 'cohort',
    'elapsed_months_g034': 'elapsed_months'
}, inplace=True)

df_g035.rename(columns={
    'cohort_g035': 'cohort',
    'elapsed_months_g035': 'elapsed_months'
}, inplace=True)

# ========== STEP 2: MERGE G034, G035 ==========

merged_34_35 = pd.merge(
    df_g034, df_g035,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in merged_34_35.columns:
    if merged_34_35[col].dtype in [float, int]:
        merged_34_35[col].fillna(0, inplace=True)
    else:
        merged_34_35[col].fillna('', inplace=True)

# ========== STEP 3: RENAME FOR ONLY THE COLUMNS WE NEED ==========

numeric_columns = ['net_revenue', 'total_cac', 'total_users', 'transaction_period_ltv_cac']

for col in numeric_columns:
    old_34 = col + '_g034'
    old_35 = col + '_g035'
    new_34 = col + '_34'
    new_35 = col + '_35'
    if old_34 in merged_34_35.columns:
        merged_34_35.rename(columns={old_34: new_34}, inplace=True)
    if old_35 in merged_34_35.columns:
        merged_34_35.rename(columns={old_35: new_35}, inplace=True)

# ========== STEP 4: READ & MERGE G036 ==========

df_g036 = pd.read_csv('G036_Dataset_12_2024.csv')
df_g036 = df_g036.add_suffix('_g036')

df_g036.rename(columns={
    'cohort_g036': 'cohort',
    'elapsed_months_g036': 'elapsed_months'
}, inplace=True)

final_merged = pd.merge(
    merged_34_35,
    df_g036,
    on=['cohort', 'elapsed_months'],
    how='outer'
)

# Fill missing numeric data with 0, string data with ''
for col in final_merged.columns:
    if final_merged[col].dtype in [float, int]:
        final_merged[col].fillna(0, inplace=True)
    else:
        final_merged[col].fillna('', inplace=True)

# Rename X_g036 -> X_36 for the numeric columns
for col in numeric_columns:
    old_36 = col + '_g036'
    new_36 = col + '_36'
    if old_36 in final_merged.columns:
        final_merged.rename(columns={old_36: new_36}, inplace=True)

# ========== STEP 5: CREATE SUM COLUMNS FOR (34 + 35) ==========

# Sum-based columns
for col in ['net_revenue', 'total_cac', 'total_users']:
    col_34 = col + '_34'
    col_35 = col + '_35'
    sum_col = col + '_34_35_sum'
    final_merged[sum_col] = (
        final_merged[col_34] +
        final_merged[col_35]
    )

# Calculate the combined ratio for 'transaction_period_ltv_cac' (net_revenue / total_cac)
final_merged['transaction_period_ltv_cac_34_35'] = (
    final_merged['net_revenue_34_35_sum'] / 
    final_merged['total_cac_34_35_sum']
).replace([float('inf'), -float('inf')], 0).fillna(0)

# ========== STEP 6: BUILD THE FINAL LIST OF COLUMNS ==========

output_columns = ['cohort', 'elapsed_months']

# Add numeric columns to the output list
for col in numeric_columns:
    if col != 'transaction_period_ltv_cac':  # Sum-based columns
        output_columns.append(col + '_34')
        output_columns.append(col + '_35')
        output_columns.append(col + '_34_35_sum')
        output_columns.append(col + '_36')
    else:  # For ratio-based columns
        output_columns.append(col + '_34_35')
        output_columns.append(col + '_36')

final_output = final_merged[output_columns].copy()

# ========== STEP 7: SAVE THE FINAL CSV ==========

final_output.to_csv('G034_G035_G036_values.csv', index=False)
print("Done! Check 'G034_G035_G036_values.csv' for the combined values.")


In [ ]:
import pandas as pd

# Function to process cohort column
def correct_cohort_format(file_path, output_path):
    # Read the file
    df = pd.read_csv(file_path)
    
    # Convert cohort to datetime and format as yyyy-mm
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y').dt.strftime('%Y-%m')
    
    # Save the updated file
    df.to_csv(output_path, index=False)
    print(f"Cohort column corrected and saved to {output_path}")

# Correct cohort column for the first CSV (27, 28, 29, 30)
correct_cohort_format('G027_G028_G029_G030_values.csv', 'G027_G028_G029_G030_values_corrected.csv')

# Correct cohort column for the second CSV (34, 35, 36)
correct_cohort_format('G034_G035_G036_values.csv', 'G034_G035_G036_values_corrected.csv')


In [ ]:
import pandas as pd

# Replace these with the actual paths/names of your two final CSVs
csv_file_1 = "G027_G028_G029_G030_values_corrected.csv"
csv_file_2 = "G034_G035_G036_values_corrected.csv"

# Read the two CSVs into dataframes
df1 = pd.read_csv(csv_file_1)
df2 = pd.read_csv(csv_file_2)

# Now write them to an Excel file with two sheets
output_excel_file = "final_combined.xlsx"

with pd.ExcelWriter(output_excel_file) as writer:
    # Write df1 to sheet named "Games_27_30"
    df1.to_excel(writer, sheet_name="Games_27_30", index=False)
    
    # Write df2 to sheet named "Games_34_36"
    df2.to_excel(writer, sheet_name="Games_34_36", index=False)

print(f"Done! Check '{output_excel_file}' for the two sheets.")
